# PS S6E4 - exp_20260410_030_cat_formula6_min

目的:
 - Chris の "Original Data's Exact Formula" で示唆された 6重要特徴を使って
   formula-inspired な最小 CatBoost モデルを作る
 - 既存主力 (018 / 024 / 025 / 026) とは違う思想の別人格候補になるか確認する

今回使うもの:
 - 6重要特徴
 - threshold features
 - high_score / low_score / formula_score
 - CatBoost

今回やらないもの:
 - 全列使用
 - 大量 pairwise
 - Optuna
 - 過度な微調整

## Imports

In [1]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

try:
    import torch
except Exception:
    torch = None

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260410_030_cat_formula6_min"
    EXP_NAME = "cat_formula6_min"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIGINAL_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_FOLDS = 5
    N_CLASSES = 3

    IMPORTANT_NUMS = [
        "Soil_Moisture",
        "Rainfall_mm",
        "Temperature_C",
        "Wind_Speed_kmh",
    ]

    IMPORTANT_CATS = [
        "Crop_Growth_Stage",
        "Mulching_Used",
    ]

    USED_COLS = IMPORTANT_NUMS + IMPORTANT_CATS

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 6,
        "grow_policy": "SymmetricTree",
        "random_seed": SEED,
        "early_stopping_rounds": 150,
        "verbose": 200,
    }

    BIAS_GRID = [
        [1.0, 1.0, 1.0],
        [1.2, 1.2, 1.2],
        [1.5, 1.5, 1.5],
        [1.5, 1.3, 1.8],
        [1.7, 1.5, 2.3],
        [1.8, 1.5, 2.4],
    ]

## utility

In [3]:
def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        try:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass

def maybe_set_catboost_device(params: dict) -> dict:
    params = params.copy()
    use_gpu = False
    if torch is not None:
        try:
            use_gpu = torch.cuda.is_available()
        except Exception:
            use_gpu = False

    if use_gpu:
        params["task_type"] = "GPU"
        params["devices"] = "0"
    else:
        params["task_type"] = "CPU"
        params.pop("devices", None)
    return params

def balanced_acc_score_mc(y_true, proba_or_pred):
    arr = np.asarray(proba_or_pred)
    if arr.ndim == 2:
        pred = np.argmax(arr, axis=1)
    else:
        pred = arr
    return float(balanced_accuracy_score(y_true, pred))

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-15, None)
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum == 0, 1.0, row_sum)
    return p / row_sum

def apply_class_weights_to_proba(proba, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    adjusted = proba * class_weights[None, :]
    return normalize_proba(adjusted)

def search_best_bias_grid(oof_proba, y_true, bias_grid):
    best_score = balanced_acc_score_mc(y_true, oof_proba)
    best_weights = np.ones(CFG.N_CLASSES, dtype=float)

    for cw in bias_grid:
        cw = np.asarray(cw, dtype=float)
        adjusted = apply_class_weights_to_proba(oof_proba, cw)
        score = balanced_acc_score_mc(y_true, adjusted)
        if score > best_score:
            best_score = score
            best_weights = cw.copy()

    return best_weights, best_score

seed_everything(CFG.SEED)
catboost_params = maybe_set_catboost_device(CFG.CATBOOST_PARAMS)
print(catboost_params)

{'loss_function': 'MultiClass', 'eval_metric': 'TotalF1:average=Macro', 'iterations': 3000, 'learning_rate': 0.03, 'depth': 6, 'grow_policy': 'SymmetricTree', 'random_seed': 42, 'early_stopping_rounds': 150, 'verbose': 200, 'task_type': 'GPU', 'devices': '0'}


## load data

In [4]:
train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
orig_df = pd.read_csv(CFG.ORIGINAL_PATH)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)
print("orig_df :", orig_df.shape)

target2idx = {v: i for i, v in enumerate(train_df[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train_df[CFG.TARGET_COL] = train_df[CFG.TARGET_COL].map(target2idx)
orig_df[CFG.TARGET_COL] = orig_df[CFG.TARGET_COL].map(target2idx)

display(train_df.head())
print(target2idx)

train_df: (630000, 21)
test_df : (270000, 20)
orig_df : (10000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


{'Low': 0, 'Medium': 1, 'High': 2}


## formula-inspired features

In [5]:
def add_formula_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # threshold features from Chris discussion
    d["soil_lt_25"] = (d["Soil_Moisture"] < 25).astype("int8")
    d["rain_lt_300"] = (d["Rainfall_mm"] < 300).astype("int8")
    d["temp_gt_30"] = (d["Temperature_C"] > 30).astype("int8")
    d["wind_gt_10"] = (d["Wind_Speed_kmh"] > 10).astype("int8")

    # categorical indicator features
    d["is_harvest"] = (d["Crop_Growth_Stage"].astype(str) == "Harvest").astype("int8")
    d["is_sowing"] = (d["Crop_Growth_Stage"].astype(str) == "Sowing").astype("int8")
    d["is_flowering"] = (d["Crop_Growth_Stage"].astype(str) == "Flowering").astype("int8")
    d["is_vegetative"] = (d["Crop_Growth_Stage"].astype(str) == "Vegetative").astype("int8")

    d["mulch_yes"] = (d["Mulching_Used"].astype(str) == "Yes").astype("int8")
    d["mulch_no"] = (d["Mulching_Used"].astype(str) == "No").astype("int8")

    # simple score from discussion
    d["high_score"] = (
        2 * d["soil_lt_25"]
        + 2 * d["rain_lt_300"]
        + 1 * d["temp_gt_30"]
        + 1 * d["wind_gt_10"]
    ).astype("int8")

    d["low_score"] = (
        2 * d["is_harvest"]
        + 2 * d["is_sowing"]
        + 1 * d["mulch_yes"]
    ).astype("int8")

    d["formula_score"] = (d["high_score"] - d["low_score"]).astype("int8")

    # a few light interactions, still formula-centered
    d["score_abs"] = np.abs(d["formula_score"]).astype("int8")
    d["soil_rain_combo"] = (d["soil_lt_25"] + d["rain_lt_300"]).astype("int8")
    d["temp_wind_combo"] = (d["temp_gt_30"] + d["wind_gt_10"]).astype("int8")

    return d

train_fe = add_formula_features(train_df)
test_fe = add_formula_features(test_df)
orig_fe = add_formula_features(orig_df)

## feature lists

In [6]:
BASE_FEATURES = CFG.USED_COLS

FORMULA_NUM_FEATURES = [
    "soil_lt_25",
    "rain_lt_300",
    "temp_gt_30",
    "wind_gt_10",
    "is_harvest",
    "is_sowing",
    "is_flowering",
    "is_vegetative",
    "mulch_yes",
    "mulch_no",
    "high_score",
    "low_score",
    "formula_score",
    "score_abs",
    "soil_rain_combo",
    "temp_wind_combo",
]

FEATURE_COLS = BASE_FEATURES + FORMULA_NUM_FEATURES
CAT_FEATURE_COLS = CFG.IMPORTANT_CATS

print("n_features =", len(FEATURE_COLS))
print("cat_features =", CAT_FEATURE_COLS)
print(FEATURE_COLS)

n_features = 22
cat_features = ['Crop_Growth_Stage', 'Mulching_Used']
['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Wind_Speed_kmh', 'Crop_Growth_Stage', 'Mulching_Used', 'soil_lt_25', 'rain_lt_300', 'temp_gt_30', 'wind_gt_10', 'is_harvest', 'is_sowing', 'is_flowering', 'is_vegetative', 'mulch_yes', 'mulch_no', 'high_score', 'low_score', 'formula_score', 'score_abs', 'soil_rain_combo', 'temp_wind_combo']


## optional sanity check against original formula structure

In [7]:
orig_formula_pred = np.where(
    orig_fe["formula_score"].values <= 0, 0,
    np.where(orig_fe["formula_score"].values <= 3, 1, 2)
)

orig_formula_ba = balanced_acc_score_mc(orig_fe[CFG.TARGET_COL].values, orig_formula_pred)
print("original formula-based BA =", orig_formula_ba)

original formula-based BA = 1.0


## CV main

In [8]:
X = train_fe[FEATURE_COLS].copy()
y = train_fe[CFG.TARGET_COL].values.astype(int)
X_test = test_fe[FEATURE_COLS].copy()

oof_proba = np.zeros((len(X), CFG.N_CLASSES), dtype=float)
test_preds = np.zeros((len(X_test), CFG.N_CLASSES), dtype=float)
fold_scores = []
feature_importances = []

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print("=" * 100)
    print(f"Fold {fold}/{CFG.N_FOLDS}")
    print("=" * 100)

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y[train_idx], y[val_idx]

    cat_idx = [X_train.columns.get_loc(c) for c in CAT_FEATURE_COLS]

    train_pool = Pool(X_train, y_train, cat_features=cat_idx)
    val_pool = Pool(X_val, y_val, cat_features=cat_idx)
    test_pool = Pool(X_test, cat_features=cat_idx)

    model = CatBoostClassifier(**catboost_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_pred = model.predict_proba(val_pool)
    test_pred = model.predict_proba(test_pool)

    oof_proba[val_idx] = val_pred
    test_preds += test_pred / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val, val_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")

    fi = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importances.append(fi)

Fold 1/5
0:	learn: 0.9693231	test: 0.9693452	best: 0.9693452 (0)	total: 262ms	remaining: 13m 6s
200:	learn: 0.9696247	test: 0.9694876	best: 0.9697623 (180)	total: 2.91s	remaining: 40.5s
400:	learn: 0.9701081	test: 0.9700175	best: 0.9701488 (310)	total: 5.43s	remaining: 35.2s
bestTest = 0.9701487848
bestIteration = 310
Shrink model to first 311 iterations.
fold_ba = 0.959503
Fold 2/5
0:	learn: 0.9692718	test: 0.9693467	best: 0.9693467 (0)	total: 13.3ms	remaining: 39.9s
200:	learn: 0.9694595	test: 0.9695924	best: 0.9696760 (189)	total: 2.58s	remaining: 36s
bestTest = 0.9699097407
bestIteration = 231
Shrink model to first 232 iterations.
fold_ba = 0.961284
Fold 3/5
0:	learn: 0.9693635	test: 0.9689549	best: 0.9689549 (0)	total: 12.8ms	remaining: 38.4s
200:	learn: 0.9694899	test: 0.9692607	best: 0.9693027 (184)	total: 2.54s	remaining: 35.3s
bestTest = 0.9693026569
bestIteration = 184
Shrink model to first 185 iterations.
fold_ba = 0.959513
Fold 4/5
0:	learn: 0.9692982	test: 0.9692836	best: 

## raw CV

In [9]:
raw_cv_ba = balanced_acc_score_mc(y, oof_proba)
print("raw_cv_ba =", raw_cv_ba)

raw_cv_ba = 0.9598984123192418


## bias tuning

In [10]:
best_class_weights, biased_cv_ba = search_best_bias_grid(oof_proba, y, CFG.BIAS_GRID)

oof_biased = apply_class_weights_to_proba(oof_proba, best_class_weights)
pred_biased = apply_class_weights_to_proba(test_preds, best_class_weights)

print("best_class_weights =", best_class_weights.tolist())
print("biased_cv_ba =", biased_cv_ba)
print("improvement =", biased_cv_ba - raw_cv_ba)

best_class_weights = [1.0, 1.0, 1.0]
biased_cv_ba = 0.9598984123192418
improvement = 0.0


## submission

In [11]:
submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = np.argmax(pred_biased, axis=1)
submission[CFG.TARGET_COL] = submission[CFG.TARGET_COL].map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

display(submission.head())

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## save artifacts

In [12]:
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_preds)
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", pred_biased)

feature_importance_df = pd.concat(feature_importances, axis=0, ignore_index=True)
feature_importance_mean = (
    feature_importance_df.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_df.to_csv(CFG.SAVE_DIR / "feature_importance_by_fold.csv", index=False)
feature_importance_mean.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)

feature_columns_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "is_cat_feature": [c in CAT_FEATURE_COLS for c in FEATURE_COLS],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

display(feature_importance_mean.head(30))

,feature,importance
0,formula_score,67.612091
1,score_abs,23.111036
2,low_score,2.659711
3,Rainfall_mm,1.938700
4,Soil_Moisture,1.632202
5,Temperature_C,0.929871
6,Wind_Speed_kmh,0.483930
7,temp_wind_combo,0.325298
8,rain_lt_300,0.254133
9,is_harvest,0.231750


## save cv_result.json

In [13]:
cv_result = {
    "exp_id": CFG.EXP_ID,
    "raw_cv_ba": float(raw_cv_ba),
    "biased_cv_ba": float(biased_cv_ba),
    "fold_scores": [float(x) for x in fold_scores],
    "best_class_weights": best_class_weights.tolist(),
    "n_features": len(FEATURE_COLS),
    "cat_features": CAT_FEATURE_COLS,
    "used_cols": CFG.USED_COLS,
    "formula_features": FORMULA_NUM_FEATURES,
    "original_formula_ba": float(orig_formula_ba),
}

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260410_030_cat_formula6_min


## quick summary

In [14]:
summary_df = pd.DataFrame({
    "item": [
        "raw_cv_ba",
        "biased_cv_ba",
        "improvement",
        "original_formula_ba",
        "n_features",
    ],
    "value": [
        raw_cv_ba,
        biased_cv_ba,
        biased_cv_ba - raw_cv_ba,
        orig_formula_ba,
        len(FEATURE_COLS),
    ],
})
display(summary_df)

,item,value
0,raw_cv_ba,0.959898
1,biased_cv_ba,0.959898
2,improvement,0.000000
3,original_formula_ba,1.000000
4,n_features,22.000000
